In [1]:
from pyspark.sql import functions as F

df_silver_page = (
    spark.read
        .format("delta")
        .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Silver.Lakehouse/Tables/dbo/dim_page")
)

print("Silver dim_page rows:", df_silver_page.count())
print("Silver dim_page columns:", len(df_silver_page.columns))

display(df_silver_page.limit(10))

StatementMeta(, 2b22e09e-199a-4a21-b42b-2e0d67f06ce5, 3, Finished, Available, Finished, False)

Silver dim_page rows: 500
Silver dim_page columns: 7


SynapseWidget(Synapse.DataFrame, 2f01c71f-8c1d-4338-9947-ecf9d58353bd)

In [2]:
from pyspark.sql.window import Window

window_page = Window.orderBy("page_id")

df_gold_page = (
    df_silver_page
    .withColumn(
        "page_key",
        F.row_number().over(window_page)
    )
)

display(
    df_gold_page
    .orderBy("page_key")
    .limit(10)
)

StatementMeta(, 2b22e09e-199a-4a21-b42b-2e0d67f06ce5, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e389d3df-e870-484c-9940-2974b69d7287)

In [3]:
(
    df_gold_page.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dim_page")
)

StatementMeta(, 2b22e09e-199a-4a21-b42b-2e0d67f06ce5, 5, Finished, Available, Finished, False)

In [4]:
df_gold_page_check = spark.read.table("dim_page")

print("Gold dim_page rows:", df_gold_page_check.count())
print("Gold dim_page columns:", len(df_gold_page_check.columns))

display(
    df_gold_page_check
    .orderBy("page_key")
    .limit(10)
)

StatementMeta(, 2b22e09e-199a-4a21-b42b-2e0d67f06ce5, 6, Finished, Available, Finished, False)

Gold dim_page rows: 500
Gold dim_page columns: 8


SynapseWidget(Synapse.DataFrame, 8612db4f-ba35-4cd0-9a48-decd37fb5920)